# Core gateway deployment

Deploy the central **Core Gateway** that all project teams connect to.

## What gets deployed

| Resource | Type | Purpose |
|----------|------|---------|
| `aif-core-{suffix}` | Foundry account (eastus2) | Primary core - shared chat + embedding models for all teams |
| `aif-research-{suffix}` | Foundry account (norwayeast) | Research hub - reasoning/research models |
| `aif-oss-{suffix}` | Foundry account (westus3) | OSS hub - open source/open-weights models |
| `apim-foundry-{suffix}` | API Management (BasicV2) | Gateway with rate limiting and managed identity auth |
| `stfoundry{suffix}` | Storage account | Shared storage |
| `project-admin-{suffix}` | Foundry project (child of aif-core) | Admin project - hosts centrally-managed agents, evaluations, observability, and load-gen |

## Model deployments

| Model | Hub | Purpose |
|-------|-----|---------|
| `gpt-4.1-mini` | aif-core | General purpose chat for all teams |
| `text-embedding-3-large` | aif-core | Vector embeddings for search |
| `o3-deep-research` | aif-research | Advanced reasoning (only available in norwayeast) |
| `Phi-4` | aif-oss | Open-weights model from Microsoft (westus3) |

## Naming conventions

Resources follow [Azure CAF abbreviations](https://learn.microsoft.com/en-us/azure/cloud-adoption-framework/ready/azure-best-practices/resource-abbreviations):
- `aif` = Foundry account (`Microsoft.CognitiveServices/accounts` kind `AIServices`)
- `apim` = API Management
- `st` = Storage account (no hyphens allowed)
- `rg` = Resource group

Foundry account names describe **role**, not model names or regions:
- `aif-core` - the default endpoint all teams use; models here are interchangeable
- `aif-research` - reasoning/research capability; region is an infrastructure detail
- `aif-oss` - open source/open-weights model tier; name stays valid if the specific model changes

## Multi-region pattern

Some models are only available in specific regions (o3-deep-research → norwayeast, Phi-4 → westus3). Rather than exposing multiple endpoints to consumers, APIM provides a **single gateway URL** and routes requests to the correct regional backend based on the deployment name in the URL path.

## Step 1: Login to Azure
Be sure you log into Azure to authenticate first, e.g., 

#!az login

#!az login --use-device-code

## Step 2: Set variables

In [1]:
import subprocess, hashlib

# Derive a stable 6-char suffix from the subscription ID
# Used in both the resource group name and all resource names
SUB_ID = subprocess.run('az account show --query id -o tsv', shell=True, capture_output=True, text=True).stdout.strip()
SUFFIX = hashlib.sha256((SUB_ID + 'v2').encode()).hexdigest()[:6]

RG = f"rg-foundry-core-{SUFFIX}"
LOCATION = "eastus2"

print(f"Suffix:         {SUFFIX}")
print(f"Resource Group: {RG}")

Suffix:         c2676f
Resource Group: rg-foundry-core-c2676f


## Step 3: Create resource group

In [2]:
!az group create -n "{RG}" -l "{LOCATION}" -o table

Location    Name
----------  ----------------------
eastus2     rg-foundry-core-c2676f


## Step 4: Deploy infrastructure

⏱️ Takes ~5-10 minutes (APIM is slow)

In [11]:
import subprocess, json, base64

# Get principal ID from cached JWT token (avoids graph.microsoft.com network call)
token = subprocess.run('az account get-access-token --query accessToken -o tsv', shell=True, capture_output=True, text=True).stdout.strip()
payload = token.split('.')[1] + '=='  # add padding
PRINCIPAL_ID = json.loads(base64.b64decode(payload))['oid']
print(f"Principal ID: {PRINCIPAL_ID}")
print(f"Suffix:       {SUFFIX}")

!az deployment group create -g "{RG}" --template-file main.bicep -p deployerPrincipalId="{PRINCIPAL_ID}" suffix="{SUFFIX}" -o table

Principal ID: 00000000-0000-0000-0000-000000000000
Suffix:       c2676f
=A new Bicep release is available: v0.43.8. Upgrade now by running "az bicep upgrade".
Name    State      Timestamp                         Mode         ResourceGroup
------  ---------  --------------------------------  -----------  ----------------------
main    Succeeded  2026-05-10T18:44:43.868372+00:00  Incremental  rg-foundry-core-c2676f


## Step 5: Get outputs

In [12]:
import subprocess, json
from pathlib import Path

r = subprocess.run(f'az deployment group show -g "{RG}" -n main --query properties.outputs -o json', shell=True, capture_output=True, text=True)
out = json.loads(r.stdout)

ENDPOINT = out['aiEndpoint']['value']
GATEWAY_URL = out['apimUrl']['value']
APIM_NAME = out['apimName']['value']
APIM_SUB_NAME = out['apimSubscriptionName']['value']
CHAT_MODEL = out['chatModelName']['value']
EMBEDDING_MODEL = out['embeddingModelName']['value']
RESEARCH_MODEL = out['researchModelName']['value']
RESEARCH_ENDPOINT = out['researchHubEndpoint']['value']
OSS_MODEL = out['ossModelName']['value']
OSS_ENDPOINT = out['ossHubEndpoint']['value']
ADMIN_PROJECT = out['adminProjectName']['value']
ADMIN_PROJECT_ENDPOINT = out['adminProjectEndpoint']['value']

# Get APIM subscription key
sub_id = subprocess.run('az account show --query id -o tsv', shell=True, capture_output=True, text=True).stdout.strip()
key_cmd = f'az rest --method POST --uri "https://management.azure.com/subscriptions/{sub_id}/resourceGroups/{RG}/providers/Microsoft.ApiManagement/service/{APIM_NAME}/subscriptions/{APIM_SUB_NAME}/listSecrets?api-version=2024-06-01-preview" --query primaryKey -o tsv'
GATEWAY_KEY = subprocess.run(key_cmd, shell=True, capture_output=True, text=True).stdout.strip()

print(f"Shared Endpoint:   {ENDPOINT}")
print(f"Research Endpoint: {RESEARCH_ENDPOINT}")
print(f"OSS Endpoint:      {OSS_ENDPOINT}")
print(f"Gateway URL:       {GATEWAY_URL}")
print(f"Gateway Key:       {GATEWAY_KEY[:2]}... (hidden)")
print(f"Chat Model:        {CHAT_MODEL}")
print(f"Embedding Model:   {EMBEDDING_MODEL}")
print(f"Research Model:    {RESEARCH_MODEL}")
print(f"OSS Model:         {OSS_MODEL}")
print(f"Admin Project:     {ADMIN_PROJECT}")
print(f"Admin Endpoint:    {ADMIN_PROJECT_ENDPOINT}")

# Merge outputs into .env file in repo root (preserves existing keys)
repo_root = Path(subprocess.run('git rev-parse --show-toplevel', shell=True, capture_output=True, text=True).stdout.strip())
env_file = repo_root / '.env'

existing = {}
if env_file.exists():
    for line in env_file.read_text().splitlines():
        if '=' in line and not line.startswith('#'):
            k, _, v = line.partition('=')
            existing[k.strip()] = v.strip()

# Remove stale keys
for stale in ('MODEL_NAME', 'DEEP_RESEARCH_MODEL', 'AI_ENDPOINT', 'APIM_URL', 'APIM_KEY', 'GATEWAY_KEY'):
    existing.pop(stale, None)

existing.update({
    'CORE_ENDPOINT': ENDPOINT,
    'GATEWAY_URL': GATEWAY_URL,
    'ALPHA_GATEWAY_KEY': GATEWAY_KEY,
    'CHAT_MODEL': CHAT_MODEL,
    'EMBEDDING_MODEL': EMBEDDING_MODEL,
    'RESEARCH_MODEL': RESEARCH_MODEL,
    'OSS_MODEL': OSS_MODEL,
    'ADMIN_FOUNDRY_PROJECT': ADMIN_PROJECT,
    'ADMIN_FOUNDRY_PROJECT_ENDPOINT': ADMIN_PROJECT_ENDPOINT,
})

env_file.write_text('\n'.join(f'{k}={v}' for k, v in existing.items()) + '\n')
print(f"\n✅ Outputs saved to {env_file}")

Shared Endpoint:   https://aif-core-c2676f.cognitiveservices.azure.com/
Research Endpoint: https://aif-research-c2676f.cognitiveservices.azure.com/
OSS Endpoint:      https://aif-oss-c2676f.cognitiveservices.azure.com/
Gateway URL:       https://apim-foundry-c2676f.azure-api.net/openai
Gateway Key:       83... (hidden)
Chat Model:        gpt-4.1-mini
Embedding Model:   text-embedding-3-large
Research Model:    o3-deep-research
OSS Model:         Phi-4
Admin Project:     project-admin-c2676f
Admin Endpoint:    https://aif-core-c2676f.services.ai.azure.com/api/projects/project-admin-c2676f

✅ Outputs saved to <repo-root>/.env


## Step 6: Test the chat model
Dependencies are managed via pyproject.toml - run `uv sync` in the repo root if packages are missing

In [5]:
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

token_provider = get_bearer_token_provider(DefaultAzureCredential(), "https://cognitiveservices.azure.com/.default")

client = AzureOpenAI(
    azure_endpoint=ENDPOINT,
    azure_ad_token_provider=token_provider,
    api_version="2024-10-21"
)

response = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": "Say hello!"}]
)

print("✅ Working!")
print(response.choices[0].message.content)

✅ Working!
Hello! How can I assist you today?


## Step 7: Test the embedding model

In [6]:
import requests

# Test embedding model via APIM gateway
response = requests.post(
    f"{GATEWAY_URL}/deployments/{EMBEDDING_MODEL}/embeddings?api-version=2024-10-21",
    headers={"api-key": GATEWAY_KEY, "Content-Type": "application/json"},
    json={"input": "Hello, space!", "model": EMBEDDING_MODEL}
)
response.raise_for_status()

embedding = response.json()["data"][0]["embedding"]
print(f"✅ Embedding model working!")
print(f"   Model: {EMBEDDING_MODEL}")
print(f"   Dimensions: {len(embedding)}")
print(f"   Sample: [{embedding[0]:.6f}, {embedding[1]:.6f}, {embedding[2]:.6f}, ...]")

✅ Embedding model working!
   Model: text-embedding-3-large
   Dimensions: 3072
   Sample: [-0.029556, -0.029648, -0.016464, ...]


## Step 8: Test the research model via APIM

The research model is routed through APIM to the research hub backend automatically.

In [7]:
from openai import AzureOpenAI

# Test research model via APIM gateway (routes to research hub backend)
research_client = AzureOpenAI(
    azure_endpoint=GATEWAY_URL.replace('/openai', ''),
    api_key=GATEWAY_KEY,
    api_version="2024-12-01-preview",
    timeout=120
)

response = research_client.chat.completions.create(
    model=RESEARCH_MODEL,
    messages=[{"role": "user", "content": "What is 2+2? Reply with just the number."}]
)

print(f"✅ Research model working via APIM!")
print(f"   Model: {RESEARCH_MODEL}")
print(f"   Response: {response.choices[0].message.content}")

✅ Research model working via APIM!
   Model: o3-deep-research
   Response: 4


In [8]:
#!az group delete -n "{RG}" --yes --no-wait

---

## Troubleshooting: soft delete

Two resource types deployed here have **soft delete** enabled by default. When you delete the resource group they are retained in a soft-deleted state and must be explicitly purged before redeploying with the same names.

### Cognitive Services (Foundry accounts)

Retained for **48 hours**. Redeploy fails with `FlagMustBeSetForRestore`.

```
FlagMustBeSetForRestore: An existing resource ... has been soft-deleted.
If you don't want to restore existing resource, please purge it first.
```

List and purge with `az cognitiveservices account list-deleted` / `az cognitiveservices account purge` (see cells below).

### API Management

Retained for **48 hours**. Redeploy fails with `ServiceAlreadyExistsInSoftDeletedState`.

```
ServiceAlreadyExistsInSoftDeletedState: Api service apim-foundry-xxx was soft-deleted.
In order to create the new service with the same name, you have to either undelete or purge it.
```

List and purge with `az apim deletedservice list` / `az apim deletedservice purge` (see cells below).

In [9]:
# List all soft-deleted Cognitive Services accounts and APIM services in the subscription
!az cognitiveservices account list-deleted -o table
!az apim deletedservice list -o table

Kind        Location       Name
----------  -------------  ------------------------------
AIServices  eastus2        aif-spoke-multi-gvwiex
AIServices  eastus2        aif-cu-ii5drx
AIServices  swedencentral  proj-foundry-jpb-core-resource
AIServices  swedencentral  aif-core-contoso-e84b7b
AIServices  swedencentral  aif-spk-contoso-e84b7b
AIServices  norwayeast     aif-research-contoso-e84b7b
DeletionDate                      Location        Name                 ScheduledPurgeDate                ServiceId
--------------------------------  --------------  -------------------  --------------------------------  -------------------------------------------------------------------------------------------------------------------------------------------------------
2026-05-08T18:32:22.934306+00:00  Sweden Central  apim-contoso-e84b7b  2026-05-10T18:31:29.209861+00:00  /subscriptions/00000000-0000-0000-0000-000000000000/resourceGroups/rg-contoso-core-e84b7b/providers/Microsoft.ApiManagement/serv

In [10]:
# Purge soft-deleted resources - uncomment and run as needed

# Cognitive Services (Foundry accounts)
#!az cognitiveservices account purge -l eastus2    -g "{RG}" -n "aif-core-{SUFFIX}"
#!az cognitiveservices account purge -l norwayeast -g "{RG}" -n "aif-research-{SUFFIX}"
#!az cognitiveservices account purge -l westus3    -g "{RG}" -n "aif-oss-{SUFFIX}"

# API Management
#!az apim deletedservice purge --service-name "apim-foundry-{SUFFIX}" --location "{LOCATION}"